In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from prepare_LIAS import process_all_LIAS_files
from tsfresh_features_LIAS import extract_tsfresh_features



In [3]:
WINDOW_SIZE = 1024
OVERLAP = int(WINDOW_SIZE/2)

# Call the new function to process all files
all_processed_windows_a, all_processed_windows_b, all_processed_windows_c = process_all_LIAS_files(WINDOW_SIZE, OVERLAP)

print(f"Total processed windows for 'isa': {len(all_processed_windows_a)}")
print(f"Total processed windows for 'isb': {len(all_processed_windows_b)}")
print(f"Total processed windows for 'isc': {len(all_processed_windows_c)}")

Found 162 files: ['ccs0.csv', 'ccs1.csv', 'ccs10.csv', 'ccs100.csv', 'ccs101.csv', 'ccs102.csv', 'ccs103.csv', 'ccs104.csv', 'ccs105.csv', 'ccs106.csv', 'ccs107.csv', 'ccs108.csv', 'ccs109.csv', 'ccs11.csv', 'ccs110.csv', 'ccs111.csv', 'ccs112.csv', 'ccs113.csv', 'ccs114.csv', 'ccs115.csv', 'ccs116.csv', 'ccs117.csv', 'ccs12.csv', 'ccs120.csv', 'ccs121.csv', 'ccs122.csv', 'ccs123.csv', 'ccs124.csv', 'ccs125.csv', 'ccs126.csv', 'ccs127.csv', 'ccs128.csv', 'ccs129.csv', 'ccs13.csv', 'ccs130.csv', 'ccs131.csv', 'ccs132.csv', 'ccs133.csv', 'ccs134.csv', 'ccs135.csv', 'ccs136.csv', 'ccs137.csv', 'ccs14.csv', 'ccs140.csv', 'ccs141.csv', 'ccs142.csv', 'ccs143.csv', 'ccs144.csv', 'ccs145.csv', 'ccs146.csv', 'ccs147.csv', 'ccs148.csv', 'ccs149.csv', 'ccs15.csv', 'ccs150.csv', 'ccs151.csv', 'ccs152.csv', 'ccs153.csv', 'ccs154.csv', 'ccs155.csv', 'ccs156.csv', 'ccs157.csv', 'ccs16.csv', 'ccs160.csv', 'ccs161.csv', 'ccs162.csv', 'ccs163.csv', 'ccs164.csv', 'ccs165.csv', 'ccs166.csv', 'ccs167.csv',

Processing files: 100%|██████████| 162/162 [00:04<00:00, 39.12it/s]

Total processed windows for 'isa': 4050
Total processed windows for 'isb': 4050
Total processed windows for 'isc': 4050


In [ ]:
all_processed_windows_a[0]

In [ ]:
def windows_to_dataframe(processed_windows):
    rows = []

    for window in processed_windows:
        row = window["metadata"].copy()

        for i, value in enumerate(window["data"]):
            row[f"sample_{i}"] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Step 2: Convert windows to DataFrames

df_a = windows_to_dataframe(all_processed_windows_a)

df_b = windows_to_dataframe(all_processed_windows_b)

df_c = windows_to_dataframe(all_processed_windows_c)

In [ ]:
df_a

In [ ]:
# Step 3: Extract TSFresh features

df_features_a = extract_tsfresh_features(df_a, "ISA")
df_features_b = extract_tsfresh_features(df_b, "ISB")
df_features_c = extract_tsfresh_features(df_c, "ISC")

In [ ]:
df_features_a

In [ ]:
columns_to_drop = ['value__sum_values', 'value__variance', 'value__length']

# Drop columns from each DataFrame
df_features_a = df_features_a.drop(columns=columns_to_drop, errors='ignore')
df_features_b = df_features_b.drop(columns=columns_to_drop, errors='ignore')
df_features_c = df_features_c.drop(columns=columns_to_drop, errors='ignore')

print("Dropped specified columns from df_features_a, df_features_b, and df_features_c.")
print(f"New shape of df_features_a: {df_features_a.shape}")
print(f"New shape of df_features_b: {df_features_b.shape}")
print(f"New shape of df_features_c: {df_features_c.shape}")

In [ ]:
feature_columns = [col for col in df_features_a.columns if not col in ['file_name', 'window_num', 'window_size', 'overlap', 'signal_type']]

feature_rename_map = {
    old_name: f'tsff_{i+1}'
    for i, old_name in enumerate(feature_columns)
}

def rename_df_features(df: pd.DataFrame, phase_suffix: str):
    """Rename feature columns as tsff_<phase>_<number>."""
    df_renamed = df.copy()

    new_columns = {}

    for col in df_renamed.columns:
        if col in feature_rename_map:
            feature_num = feature_rename_map[col].replace('tsff_', '')
            new_columns[col] = f"tsff_{phase_suffix}_{feature_num}"
        else:
            new_columns[col] = col

    return df_renamed.rename(columns=new_columns)

# Apply the renaming to each phase dataframe
df_features_a_renamed = rename_df_features(df_features_a, '_a')
df_features_b_renamed = rename_df_features(df_features_b, '_b')
df_features_c_renamed = rename_df_features(df_features_c, '_c')


In [ ]:
df_features_a_renamed

In [ ]:
# Define common columns for merging (metadata that should be the same across phases for a given window)
common_merge_cols = ['file_name', 'window_num', 'window_size', 'overlap']

# Start with phase A features, selecting only the necessary columns
df_merged = df_features_a_renamed[common_merge_cols + [col for col in df_features_a_renamed.columns if col.startswith('tsff_')]]

# Merge with phase B features
# Select only the features and common merge columns from df_features_b_renamed
df_b_for_merge = df_features_b_renamed[common_merge_cols + [col for col in df_features_b_renamed.columns if col.startswith('tsff_')]]
df_merged = pd.merge(df_merged, df_b_for_merge, on=common_merge_cols, how='inner')

# Merge with phase C features
# Select only the features and common merge columns from df_features_c_renamed
df_c_for_merge = df_features_c_renamed[common_merge_cols + [col for col in df_features_c_renamed.columns if col.startswith('tsff_')]]
df_merged = pd.merge(df_merged, df_c_for_merge, on=common_merge_cols, how='inner')

print(f"Merged DataFrame created with {len(df_merged)} rows and {df_merged.shape[1]} columns.")
display(df_merged.head())

In [ ]:
## ----- CREATING SOME DICTIONARIES TO MAP THE FILE NAMES, GROUPS AND LABELS (TARGET) ----##

motor_conditions_ranges = {
    "healthy_low": range(0, 18), # 18
    "one_broken_low": range(60, 68), # 8
    "two_broken_low": range(120, 128), # 8

    "healthy_med": range(20, 38),#18
    "one_broken_med": range(80, 98),#18
    "two_broken_med": range(140, 148), #8

    "healthy_high": range(40, 58), #18
    "one_broken_high": range(100, 118), #18
    "two_broken_high": range(160, 178) #18
}

def generate_valid_files_list(motor_conditions_ranges: dict) -> list:
  """
  Generates a list of valid file names based on motor condition ranges.

  Args:
    motor_conditions_ranges (dict): A dictionary where keys are condition names
                                     and values are range objects representing file indices.

  Returns:
    list: A list of file names (e.g., ['ccs0.csv', 'ccs1.csv', ...]).
  """
  valid_files = []
  for r in motor_conditions_ranges.values():
      for i in r:
          valid_files.append(f"ccs{i}.csv")
  return valid_files

def generate_file_to_class_map(motor_conditions_ranges: dict) -> dict:
  """
  Generates a mapping from file names to their corresponding class labels.

  Args:
    motor_conditions_ranges (dict): A dictionary where keys are condition names
                                     and values are range objects representing file indices.

  Returns:
    dict: A dictionary where keys are file names (e.g., 'ccs0.csv') and values
          are class labels (1 for healthy, 2 for one broken, 3 for two broken).
  """
  file_to_class_map = {}

  # Class 1: Healthy conditions
  for key in ["healthy_low", "healthy_med", "healthy_high"]:
      for i in motor_conditions_ranges[key]:
          file_to_class_map[f"ccs{i}.csv"] = 1

  # Class 2: One broken condition
  for key in ["one_broken_low", "one_broken_med", "one_broken_high"]:
      for i in motor_conditions_ranges[key]:
          file_to_class_map[f"ccs{i}.csv"] = 2

  # Class 3: Two broken conditions
  for key in ["two_broken_low", "two_broken_med", "two_broken_high"]:
      for i in motor_conditions_ranges[key]:
          file_to_class_map[f"ccs{i}.csv"] = 3

  return file_to_class_map


def generate_file_to_group_map(motor_conditions_ranges: dict) -> dict:
  """
  Generates a mapping from file names to their corresponding group names.

  Args:
    motor_conditions_ranges (dict): A dictionary where keys are group names
                                     and values are range objects representing file indices.

  Returns:
    dict: A dictionary where keys are file names (e.g., 'ccs0.csv') and values
          are group names (e.g., 'healthy_low').
  """
  file_to_group_map = {}
  for group_name, file_range in motor_conditions_ranges.items():
    for i in file_range:
        file_to_group_map[f"ccs{i}.csv"] = group_name

  return file_to_group_map

valid_files = generate_valid_files_list(motor_conditions_ranges)
file_to_class_map = generate_file_to_class_map(motor_conditions_ranges)
file_to_group_map = generate_file_to_group_map(motor_conditions_ranges)

In [ ]:
# Filtering df_merged for only those embeddings that belong to valid files
df_merged_valid = df_merged[df_merged['file_name'].isin(valid_files)].copy()
print(f"Generated {len(valid_files)} valid files:")


# Attaching target column
df_merged_valid['class'] = df_merged_valid['file_name'].map(file_to_class_map)

# Add a file group column to enable stratified splitting
df_merged_valid['file_group'] = df_merged_valid['file_name'].map(file_to_group_map)
display(df_merged_valid.head())

In [ ]:
# ---- Train-test split at file-level while stratifying on file_group ---- #
file_df = df_merged_valid[['file_name', 'file_group']].drop_duplicates()

from sklearn.model_selection import train_test_split
train_files, test_files = train_test_split(file_df["file_name"],
                                           stratify=file_df["file_group"],
                                           test_size=0.2,
                                           random_state=67)

train_df = df_merged_valid[df_merged_valid["file_name"].isin(train_files)]
test_df = df_merged_valid[df_merged_valid["file_name"].isin(test_files)]

In [ ]:
# Checking this worked.  Should give an empty set
set(train_df["file_name"]) & set(test_df["file_name"])

In [ ]:
print(train_df["file_group"].value_counts(normalize=True))
print(test_df["file_group"].value_counts(normalize=True))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify feature columns (those starting with 'moment_')
feature_cols = [col for col in train_df.columns if col.startswith('tsff_')]

# Prepare training data
X_train = train_df[feature_cols]
y_train = train_df['class']

# Prepare testing data
X_test = test_df[feature_cols]
y_test = test_df['class']

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardized using StandardScaler.")

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np # Import numpy for type hinting np.ndarray
from sklearn.linear_model import LogisticRegression

def train_svc_model(X_train: pd.DataFrame | np.ndarray, y_train: pd.Series, kernel: str = 'rbf', class_weight: str = 'balanced', random_state: int = 42) -> SVC:
  """
  Trains an SVC classifier and returns the trained model.

  Args:
    X_train (pd.DataFrame | np.ndarray): Training features.
    y_train (pd.Series): Training labels.
    kernel (str): Specifies the kernel type to be used in the algorithm.
    class_weight (str): Weights associated with classes.
    random_state (int): The seed of the pseudo random number generator.

  Returns:
    SVC: The trained SVC model.
  """
  model = SVC(kernel=kernel, class_weight=class_weight, random_state=random_state)
  print(f"Training SVM model with {kernel} kernel and balanced class weights...")
  model.fit(X_train, y_train)
  print(f"SVM model training complete with {kernel} kernel.")
  return model

def evaluate_model(y_true: pd.Series, y_pred: np.ndarray, title: str):
  """
  Evaluates a classification model by printing the classification report and confusion matrix.

  Args:
    y_true (pd.Series): True labels.
    y_pred (np.ndarray): Predicted labels.
    title (str): Title for the evaluation output.
  """
  print(f"\nClassification Report ({title}):")
  print(classification_report(y_true, y_pred))
  print(f"\nConfusion Matrix ({title}):")
  print(confusion_matrix(y_true, y_pred))

def train_log_reg_model(X_train: pd.DataFrame | np.ndarray, y_train: pd.Series,
                      solver: str = 'liblinear', penalty: str = 'l2', C: float = 1.0,
                      class_weight: str = 'balanced', random_state: int = 42, max_iter: int = 1000) -> LogisticRegression:
  """
  Trains a Logistic Regression classifier and returns the trained model.

  Args:
    X_train (pd.DataFrame | np.ndarray): Training features.
    y_train (pd.Series): Training labels.
    solver (str): Algorithm to use in the optimization problem.
    penalty (str): Used to specify the norm of the penalty.
    C (float): Inverse of regularization strength.
    class_weight (str): Weights associated with classes.
    random_state (int): The seed of the pseudo random number generator.
    max_iter (int): Maximum number of iterations for the solver to converge.

  Returns:
    LogisticRegression: The trained Logistic Regression model.
  """
  model = LogisticRegression(solver=solver, penalty=penalty, C=C,
                             class_weight=class_weight, random_state=random_state, max_iter=max_iter)
  print(f"Training Logistic Regression model with solver='{solver}', penalty='{penalty}', C={C} and balanced class weights...")
  model.fit(X_train, y_train)
  print("Logistic Regression model training complete.")
  return model

In [ ]:
# Train SVM model with RBF kernel on UNscaled data
svm_model_unscaled_rbf = train_svc_model(X_train, y_train, kernel='rbf')

In [ ]:
# Make predictions on the unscaled test set
y_pred_unscaled_rbf = svm_model_unscaled_rbf.predict(X_test)

# Evaluate the model
evaluate_model(y_test, y_pred_unscaled_rbf, "RBF Kernel SVM on Unscaled Data")

In [ ]:
# Train SVM model with RBF kernel on scaled data
svm_model_scaled_rbf = train_svc_model(X_train_scaled, y_train, kernel='rbf')

In [ ]:
# Make predictions on the scaled test set
y_pred_scaled_rbf = svm_model_scaled_rbf.predict(X_test_scaled)

# Evaluate the model
evaluate_model(y_test, y_pred_scaled_rbf, "RBF Kernel SVM on Scaled Data")